# Notebook Prerequisites

- **Kernel / Python:** Use the project's virtual environment (see `pyproject.toml`).
- **Required packages:** `xgboost`, `optuna`, `mlflow`, `pandas`, `scikit-learn`, `category_encoders`.
- **Data:** Processed datasets live at `data/processed`. Run `02_feature_eng_encoding.ipynb` first if files are missing.
- **MLflow:** Local `mlruns/` is used for tracking by default (created if missing).

In [2]:
import xgboost as xgb
print(xgb.__version__)

3.1.2


In [3]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

c:\Users\vigne\Desktop\Rml\.venv\Scripts\python.exe
3.1.2
c:\Users\vigne\Desktop\Rml\.venv\Lib\site-packages\xgboost\__init__.py


In [4]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

c:\Users\vigne\Desktop\Rml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from pathlib import Path
# ==============================================
# 2. Load processed datasets
# ==============================================
DATA_DIR = Path("../data/processed")
train_path = DATA_DIR / "feature_engineered_train.csv"
eval_path = DATA_DIR / "feature_engineered_eval.csv"

if not train_path.exists() or not eval_path.exists():
    raise FileNotFoundError(
        f"Expected processed files at {DATA_DIR.resolve()} - run `02_feature_eng_encoding.ipynb` first to create them."
    )

train_df = pd.read_csv(train_path)
eval_df  = pd.read_csv(eval_path)

# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 37)
Eval shape: (148448, 37)


In [6]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [7]:
from pathlib import Path
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder (portable file URI)
mlruns_path = Path("../mlruns").resolve()
mlruns_path.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(mlruns_path.as_uri())
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=3)

print("Best params:", study.best_trial.params)

c:\Users\vigne\Desktop\Rml\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
[I 2026-07-17 20:17:41,581] A new study created in memory with name: no-name-55685df9-79df-4728-9734-e27c5607d445
[I 2026-07-17 20:18:20,185] Trial 0 finished with value: 71374.24127918325 and parameters: {'n_estimators': 742, 'max_depth': 7, 'learning_rate': 0.174896647339998, 'subsample': 0.8450101924680163, 'colsample_bytree': 0.9286476239696888, 'min_child_weight': 1, 'gamma': 2.74842991520565, 'reg_alpha': 0.022336129393238414, 'reg_lambda': 0.

Best params: {'n_estimators': 742, 'max_depth': 7, 'learning_rate': 0.174896647339998, 'subsample': 0.8450101924680163, 'colsample_bytree': 0.9286476239696888, 'min_child_weight': 1, 'gamma': 2.74842991520565, 'reg_alpha': 0.022336129393238414, 'reg_lambda': 0.05034168503132877}


In [8]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
import joblib
from pathlib import Path

best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    # Save model as an artifact using joblib to avoid flavor serialization issues
    model_dir = Path("../models")
    model_dir.mkdir(parents=True, exist_ok=True)
    model_file = model_dir / "best_xgb_model.joblib"
    joblib.dump(best_model, model_file)
    mlflow.log_artifact(str(model_file))

Final tuned model performance:
MAE: 32018.50060849559
RMSE: 74593.20893702286
R²: 0.9570009745881471
